In [1]:
from pathlib import Path
import time

import pandas as pd
import comtradeapicall


# -----------------------------
# Find project root
# -----------------------------

def find_project_root(start_path: Path) -> Path:
    start_path = start_path.resolve()

    for path in [start_path] + list(start_path.parents):
        if (path / "data").exists() and (path / "src").exists():
            return path

    raise FileNotFoundError(
        "Could not find project root. Make sure this notebook is inside the project folder."
    )


BASE_DIR = find_project_root(Path.cwd())

RAW_DATA_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed"

CLEAN_DATA_PATH = PROCESSED_DATA_DIR / "seafood_trade_clean.csv"

RAW_2025_CHECK_PATH = RAW_DATA_DIR / "comtrade_2025_availability_raw.csv"
SUMMARY_2025_CHECK_PATH = PROCESSED_DATA_DIR / "availability_check_2025_summary.csv"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------
# Project scope
# -----------------------------

IMPORTER_COUNTRIES = {
    "USA": "842",
    "China": "156",
    "Japan": "392",
    "Spain": "724",
    "Germany": "276",
    "United Kingdom": "826",
    "Thailand": "764",
    "Viet Nam": "704",
    "Philippines": "608",
    "Côte d'Ivoire": "384",
    "Nigeria": "566",
    "Ghana": "288",
    "Egypt": "818",
    "United Arab Emirates": "784",
}

PRODUCT_GROUPS = {
    "Frozen Atlantic salmon": "030313",
    "Frozen yellowfin tuna": "030342",
    "Frozen mackerel": "030354",
    "Frozen hake": "030366",
    "Frozen shrimp/prawns": "030617",
}


# -----------------------------
# Helpers
# -----------------------------

def choose_column(df: pd.DataFrame, candidates: list[str]) -> str:
    for col in candidates:
        if col in df.columns:
            return col

    raise KeyError(f"None of these columns were found: {candidates}")


def fetch_2025_annual(importer_name, importer_code, product_name, product_code):
    print(f"Checking 2025 | {importer_name} | {product_name}")

    try:
        df = comtradeapicall.previewFinalData(
            typeCode="C",
            freqCode="A",
            clCode="HS",
            period="2025",
            reporterCode=importer_code,
            cmdCode=product_code,
            flowCode="M",
            partnerCode=None,
            partner2Code=None,
            customsCode=None,
            motCode=None,
            maxRecords=500,
            format_output="JSON",
            aggregateBy=None,
            breakdownMode="classic",
            countOnly=None,
            includeDesc=True,
        )

        if df is None or df.empty:
            return pd.DataFrame()

        df["selected_importer"] = importer_name
        df["selected_product_group"] = product_name
        df["selected_product_code"] = product_code

        return df

    except Exception as e:
        print(f"Failed: {importer_name} | {product_name}")
        print(e)
        return pd.DataFrame()


def summarize_comtrade_query(df, importer_name, product_name):
    if df is None or df.empty:
        return {
            "year": 2025,
            "importer_country": importer_name,
            "product_group": product_name,
            "has_2025_data": False,
            "rows": 0,
            "supplier_count": 0,
            "trade_value_usd_2025": 0,
            "quantity_kg_2025": 0,
        }

    supplier_col = choose_column(df, ["partnerDesc", "partner", "partnerName"])
    trade_value_col = choose_column(df, ["primaryValue", "cifvalue", "tradeValue"])
    quantity_col = choose_column(df, ["netWgt", "qty", "quantity"])

    trade_value = pd.to_numeric(df[trade_value_col], errors="coerce").fillna(0)
    quantity = pd.to_numeric(df[quantity_col], errors="coerce").fillna(0)

    return {
        "year": 2025,
        "importer_country": importer_name,
        "product_group": product_name,
        "has_2025_data": True,
        "rows": len(df),
        "supplier_count": df[supplier_col].nunique(),
        "trade_value_usd_2025": trade_value.sum(),
        "quantity_kg_2025": quantity.sum(),
    }


# -----------------------------
# Fetch 2025 annual data
# -----------------------------

all_raw_2025 = []
summary_rows = []

for importer_name, importer_code in IMPORTER_COUNTRIES.items():
    for product_name, product_code in PRODUCT_GROUPS.items():
        df_query = fetch_2025_annual(
            importer_name=importer_name,
            importer_code=importer_code,
            product_name=product_name,
            product_code=product_code,
        )

        if not df_query.empty:
            all_raw_2025.append(df_query)

        summary_rows.append(
            summarize_comtrade_query(
                df=df_query,
                importer_name=importer_name,
                product_name=product_name,
            )
        )

        time.sleep(0.4)


availability_df = pd.DataFrame(summary_rows)


if all_raw_2025:
    raw_2025_df = pd.concat(all_raw_2025, ignore_index=True)
    raw_2025_df.to_csv(RAW_2025_CHECK_PATH, index=False)
    print(f"\nSaved raw 2025 check data to: {RAW_2025_CHECK_PATH}")
else:
    print("\nNo raw 2025 data returned.")


# -----------------------------
# Compare against 2024 local clean data
# -----------------------------

if CLEAN_DATA_PATH.exists():
    clean_df = pd.read_csv(CLEAN_DATA_PATH)

    clean_2024 = clean_df[clean_df["year"] == 2024].copy()

    benchmark_2024 = (
        clean_2024.groupby(["importer_country", "product_group"], as_index=False)
        .agg(
            trade_value_usd_2024=("trade_value_usd", "sum"),
            quantity_kg_2024=("quantity_kg", "sum"),
            suppliers_2024=("supplier_country", "nunique"),
        )
    )

    availability_df = availability_df.merge(
        benchmark_2024,
        on=["importer_country", "product_group"],
        how="left",
    )

    availability_df["value_ratio_2025_vs_2024"] = (
        availability_df["trade_value_usd_2025"]
        / availability_df["trade_value_usd_2024"]
    )

    availability_df["quantity_ratio_2025_vs_2024"] = (
        availability_df["quantity_kg_2025"]
        / availability_df["quantity_kg_2024"]
    )
else:
    print("No clean 2024 benchmark file found.")
    availability_df["trade_value_usd_2024"] = None
    availability_df["quantity_kg_2024"] = None
    availability_df["suppliers_2024"] = None
    availability_df["value_ratio_2025_vs_2024"] = None
    availability_df["quantity_ratio_2025_vs_2024"] = None


availability_df.to_csv(SUMMARY_2025_CHECK_PATH, index=False)


# -----------------------------
# Usefulness decision
# -----------------------------

expected_combos = len(IMPORTER_COUNTRIES) * len(PRODUCT_GROUPS)

available_combos = availability_df["has_2025_data"].sum()
coverage_pct = available_combos / expected_combos

importers_covered = availability_df.loc[
    availability_df["has_2025_data"], "importer_country"
].nunique()

products_covered = availability_df.loc[
    availability_df["has_2025_data"], "product_group"
].nunique()

total_2025_value = availability_df["trade_value_usd_2025"].sum()

if "trade_value_usd_2024" in availability_df.columns:
    total_2024_value = availability_df["trade_value_usd_2024"].sum()
    value_ratio = total_2025_value / total_2024_value if total_2024_value > 0 else 0
else:
    total_2024_value = None
    value_ratio = None


print("\n==============================")
print("2025 ANNUAL DATA AVAILABILITY")
print("==============================")

print(f"Expected country-product combinations: {expected_combos}")
print(f"Combinations with 2025 data: {available_combos}")
print(f"Coverage: {coverage_pct:.1%}")
print(f"Importers covered: {importers_covered} / {len(IMPORTER_COUNTRIES)}")
print(f"Products covered: {products_covered} / {len(PRODUCT_GROUPS)}")
print(f"Total 2025 trade value: ${total_2025_value:,.0f}")

if value_ratio is not None:
    print(f"Total 2024 trade value benchmark: ${total_2024_value:,.0f}")
    print(f"2025 value as % of 2024: {value_ratio:.1%}")


print("\n==============================")
print("RECOMMENDATION")
print("==============================")

if coverage_pct >= 0.85 and importers_covered >= 12 and products_covered == 5:
    if value_ratio is None or value_ratio >= 0.75:
        recommendation = "USE_2025_AS_DEFAULT"
        explanation = (
            "2025 annual data looks broadly available and comparable enough "
            "to use as the default dashboard year."
        )
    else:
        recommendation = "USE_2025_WITH_CAUTION"
        explanation = (
            "Coverage is good, but total 2025 value is much lower than 2024. "
            "This may indicate partial reporting or a real market drop. Check country-level gaps."
        )

elif coverage_pct >= 0.50 and importers_covered >= 7 and products_covered >= 3:
    recommendation = "USE_2025_AS_PARTIAL_ONLY"
    explanation = (
        "2025 data exists, but coverage is not complete enough to make it the default. "
        "Keep 2024 as the default and label 2025 as partial/latest available data."
    )

else:
    recommendation = "DO_NOT_USE_2025_YET"
    explanation = (
        "2025 annual data is too incomplete for the current dashboard. "
        "Keep 2024 as the default year."
    )

print(recommendation)
print(explanation)


# -----------------------------
# Show useful tables
# -----------------------------

print("\nMissing 2025 combinations:")
missing_2025 = availability_df[~availability_df["has_2025_data"]][
    ["importer_country", "product_group"]
]
display(missing_2025)

print("\nTop 2025 country-product combinations by trade value:")
top_2025 = availability_df.sort_values(
    "trade_value_usd_2025",
    ascending=False,
)[
    [
        "importer_country",
        "product_group",
        "trade_value_usd_2025",
        "quantity_kg_2025",
        "supplier_count",
        "value_ratio_2025_vs_2024",
    ]
].head(20)

display(top_2025)

print("\nThinnest 2025 combinations compared with 2024:")
thin_2025 = availability_df[
    availability_df["has_2025_data"]
    & availability_df["value_ratio_2025_vs_2024"].notna()
].sort_values("value_ratio_2025_vs_2024")[
    [
        "importer_country",
        "product_group",
        "trade_value_usd_2025",
        "trade_value_usd_2024",
        "value_ratio_2025_vs_2024",
    ]
].head(20)

display(thin_2025)


print(f"\nSaved summary to: {SUMMARY_2025_CHECK_PATH}")

Checking 2025 | USA | Frozen Atlantic salmon
Checking 2025 | USA | Frozen yellowfin tuna
Checking 2025 | USA | Frozen mackerel
Checking 2025 | USA | Frozen hake
Checking 2025 | USA | Frozen shrimp/prawns
Checking 2025 | China | Frozen Atlantic salmon
Checking 2025 | China | Frozen yellowfin tuna
Checking 2025 | China | Frozen mackerel
Checking 2025 | China | Frozen hake
Checking 2025 | China | Frozen shrimp/prawns
Checking 2025 | Japan | Frozen Atlantic salmon
Checking 2025 | Japan | Frozen yellowfin tuna
Checking 2025 | Japan | Frozen mackerel
Checking 2025 | Japan | Frozen hake
Checking 2025 | Japan | Frozen shrimp/prawns
Checking 2025 | Spain | Frozen Atlantic salmon
Checking 2025 | Spain | Frozen yellowfin tuna
Checking 2025 | Spain | Frozen mackerel
Checking 2025 | Spain | Frozen hake
Checking 2025 | Spain | Frozen shrimp/prawns
Checking 2025 | Germany | Frozen Atlantic salmon
Checking 2025 | Germany | Frozen yellowfin tuna
Checking 2025 | Germany | Frozen mackerel
Checking 2025 |

,importer_country,product_group
5,China,Frozen Atlantic salmon
6,China,Frozen yellowfin tuna
7,China,Frozen mackerel
8,China,Frozen hake
9,China,Frozen shrimp/prawns
30,Thailand,Frozen Atlantic salmon
31,Thailand,Frozen yellowfin tuna
32,Thailand,Frozen mackerel
33,Thailand,Frozen hake
34,Thailand,Frozen shrimp/prawns



Top 2025 country-product combinations by trade value:


,importer_country,product_group,trade_value_usd_2025,quantity_kg_2025,supplier_count,value_ratio_2025_vs_2024
4,USA,Frozen shrimp/prawns,1.044328e+10,1.219856e+09,39,2.246398
19,Spain,Frozen shrimp/prawns,2.560677e+09,3.590286e+08,54,2.239774
14,Japan,Frozen shrimp/prawns,2.549596e+09,2.858134e+08,32,2.086501
29,United Kingdom,Frozen shrimp/prawns,8.908676e+08,4.686120e+07,46,2.211034
24,Germany,Frozen shrimp/prawns,8.392582e+08,8.704715e+07,72,2.714246
12,Japan,Frozen mackerel,4.182953e+08,1.217413e+08,17,2.398623
16,Spain,Frozen yellowfin tuna,3.365804e+08,1.180404e+08,39,1.810102
11,Japan,Frozen yellowfin tuna,2.615681e+08,6.713596e+07,22,3.527339
62,Egypt,Frozen mackerel,2.277639e+08,8.161059e+07,22,1.720054
57,Ghana,Frozen mackerel,2.201022e+08,1.760339e+08,45,NaN



Thinnest 2025 combinations compared with 2024:


,importer_country,product_group,trade_value_usd_2025,trade_value_usd_2024,value_ratio_2025_vs_2024
20,Germany,Frozen Atlantic salmon,5.129623e+07,3.526719e+07,1.454503
26,United Kingdom,Frozen yellowfin tuna,2.898142e+05,1.920143e+05,1.509337
64,Egypt,Frozen shrimp/prawns,1.824484e+08,1.157329e+08,1.576462
62,Egypt,Frozen mackerel,2.277639e+08,1.324167e+08,1.720054
3,USA,Frozen hake,7.486040e+06,4.287044e+06,1.746201
42,Philippines,Frozen mackerel,1.191512e+08,6.676097e+07,1.784743
16,Spain,Frozen yellowfin tuna,3.365804e+08,1.859456e+08,1.810102
1,USA,Frozen yellowfin tuna,5.637921e+07,3.072672e+07,1.834859
0,USA,Frozen Atlantic salmon,4.135546e+07,2.224091e+07,1.859432
25,United Kingdom,Frozen Atlantic salmon,4.141096e+06,2.116646e+06,1.956443



Saved summary to: A:\Fisheries Dashboard\seafood-trade-scanner\data\processed\availability_check_2025_summary.csv
